# Sets — Lesson

Sets are Python's **built-in data structure for uniqueness and fast membership testing**. They are hash-based, unordered collections that guarantee every element appears exactly once. If dictionaries map keys to values, sets are like dictionaries with **keys only** — and that single insight unlocks enormous power.

### Why sets matter

- **Interviews:** Deduplication, O(1) lookups, intersection/union problems, "contains duplicate", "two sum" lookup tables, anagram detection, finding missing/extra elements
- **Backend:** Access control lists, feature flag checks, deduplication of API inputs, set-based permission systems, rate limiting with unique IPs
- **Data engineering:** Deduplication pipelines, finding schema differences, comparing column sets between tables, data quality checks

### What you'll learn

1. What a set is and how it works internally (hash table intuition)
2. Creating sets from literals, constructors, and iterables
3. Set properties: unordered, unique, mutable, hashable-elements-only
4. Adding and removing elements
5. Membership testing — the killer feature (O(1) lookup)
6. All set operations: union, intersection, difference, symmetric difference
7. Subset, superset, and disjoint checks
8. In-place (mutating) set operations
9. Set comprehensions
10. Frozensets (immutable sets)
11. When to use sets vs lists vs dicts
12. Common set patterns for interviews and production
13. Performance and time complexity
14. Gotchas and edge cases

---
## 1. What Is a Set? — Intuition

Think of a set like a **bag of unique tokens**. You can toss items in, but:

- **No duplicates allowed.** If you try to add something that's already there, the set just ignores it.
- **No ordering.** Items don't have positions — there's no index 0, 1, 2. You can't slice a set.
- **Blazing fast lookups.** Checking "is this item in the bag?" is O(1) average — the same speed whether the set has 10 items or 10 million.

Under the hood, Python sets use a **hash table** — the exact same mechanism that powers dictionary keys. Each element is hashed to a position in an internal array. That's why:

1. Lookup is O(1) — hash directly to the bucket
2. Elements must be **hashable** (immutable) — you can't put a list inside a set
3. Order is not guaranteed — hash positions don't correspond to insertion order

If you understand how dict keys work, you already understand sets. A set is essentially `dict.keys()` without the values.

---
## 2. Creating Sets

There are several ways to create sets in Python. Pay close attention to the **empty set gotcha** — it trips up nearly everyone.

### 2a. Set literal with curly braces

In [ ]:
# Curly braces with values creates a set
fruits = {"apple", "banana", "cherry"}
print(fruits)        # Order may vary!
print(type(fruits))  # <class 'set'>

### 2b. The empty set gotcha

**CRITICAL:** Empty curly braces `{}` create a **dict**, not a set. You must use `set()` for an empty set. This is one of the most common Python mistakes.

In [ ]:
# THIS IS A DICT, NOT A SET!
not_a_set = {}
print(type(not_a_set))  # <class 'dict'>

# THIS is how you make an empty set
empty_set = set()
print(type(empty_set))  # <class 'set'>
print(len(empty_set))   # 0

### 2c. `set()` constructor from iterables

The `set()` constructor accepts any iterable — a list, tuple, string, range, generator, even another set. Duplicates are automatically removed.

In [ ]:
# From a list (duplicates removed)
nums = set([1, 2, 3, 2, 1, 4])
print(nums)  # {1, 2, 3, 4}

# From a string (each character becomes an element)
chars = set("hello")
print(chars)  # {'h', 'e', 'l', 'o'} — only one 'l'!

# From a tuple
t = set((10, 20, 30, 10))
print(t)  # {10, 20, 30}

# From a range
r = set(range(5))
print(r)  # {0, 1, 2, 3, 4}

# From dict keys (only keys, not values)
d = {"a": 1, "b": 2, "c": 3}
s = set(d)
print(s)  # {'a', 'b', 'c'}

### 2d. Set with mixed types

Sets can hold elements of different types, as long as each element is **hashable** (immutable). Integers, floats, strings, tuples, booleans, `None` — all fine. Lists and dicts — not allowed.

In [ ]:
# Mixed types — all hashable
mixed = {1, "hello", 3.14, True, None, (1, 2)}
print(mixed)

# Watch out: True == 1 and False == 0 in Python
# So {1, True} has only ONE element
tricky = {1, True}
print(tricky)      # {1}
print(len(tricky)) # 1

### 2e. What you CANNOT put in a set

Unhashable (mutable) objects cause a `TypeError`. This includes lists, dicts, and sets themselves.

In [ ]:
# Tuples are fine (immutable)
s = {(1, 2), (3, 4)}
print(s)  # {(1, 2), (3, 4)}

# Lists are NOT hashable — this will raise TypeError
try:
    bad = {[1, 2], [3, 4]}
except TypeError as e:
    print(f"Error: {e}")  # unhashable type: 'list'

# Dicts are NOT hashable
try:
    bad = {{"a": 1}}
except TypeError as e:
    print(f"Error: {e}")  # unhashable type: 'dict'

# Sets are NOT hashable (use frozenset instead)
try:
    bad = {{1, 2}}
except TypeError as e:
    print(f"Error: {e}")  # unhashable type: 'set'

---
## 3. Set Properties — The Four Rules

Every set in Python has four fundamental properties:

| Property | What it means | Consequence |
|----------|--------------|-------------|
| **Unordered** | Elements have no position | No indexing, no slicing, iteration order not guaranteed |
| **Unique** | No duplicate elements | Adding a duplicate is silently ignored |
| **Mutable** | Can add/remove elements after creation | Use `frozenset` if you need immutable |
| **Hashable elements only** | Elements must be immutable | No lists, dicts, or sets inside a set |

In [ ]:
# UNORDERED — no indexing
s = {10, 20, 30}
try:
    print(s[0])  # TypeError!
except TypeError as e:
    print(f"No indexing: {e}")

# UNIQUE — duplicates silently dropped
s = {1, 1, 2, 2, 3, 3}
print(s)       # {1, 2, 3}
print(len(s))  # 3

# MUTABLE — can change after creation
s = {1, 2}
s.add(3)
print(s)  # {1, 2, 3}

---
## 4. Adding Elements

Two methods for adding: `add()` for a single element, `update()` for multiple elements from an iterable.

### 4a. `add()` — add a single element

In [ ]:
colors = {"red", "green"}

# Add a new element
colors.add("blue")
print(colors)  # {'red', 'green', 'blue'}

# Adding a duplicate — no error, just ignored
colors.add("red")
print(colors)  # {'red', 'green', 'blue'} — unchanged
print(len(colors))  # 3

### 4b. `update()` — add multiple elements from an iterable

`update()` accepts any iterable — list, tuple, string, another set, range, etc. It adds each element individually.

In [ ]:
s = {1, 2}

# Update from a list
s.update([3, 4, 5])
print(s)  # {1, 2, 3, 4, 5}

# Update from multiple iterables at once
s.update([10, 20], {30, 40}, (50,))
print(s)  # {1, 2, 3, 4, 5, 10, 20, 30, 40, 50}

# Update from a string (each character added)
chars = set()
chars.update("abc")
print(chars)  # {'a', 'b', 'c'}

# GOTCHA: add() vs update() with a string
s1 = set()
s1.add("hello")     # Adds the WHOLE string as one element
print(s1)            # {'hello'}

s2 = set()
s2.update("hello")  # Adds EACH CHARACTER as separate elements
print(s2)            # {'h', 'e', 'l', 'o'}

---
## 5. Removing Elements

Python gives you four ways to remove from a set, each with different behavior on missing elements.

### 5a. `remove()` — remove or raise KeyError

Use when you're **sure** the element exists. Raises `KeyError` if not found.

In [ ]:
s = {1, 2, 3, 4, 5}

s.remove(3)
print(s)  # {1, 2, 4, 5}

# Removing a non-existent element → KeyError
try:
    s.remove(99)
except KeyError as e:
    print(f"KeyError: {e}")  # KeyError: 99

### 5b. `discard()` — remove or do nothing

The **safe** removal method. If the element doesn't exist, it silently does nothing. No error. This is usually what you want in production code.

In [ ]:
s = {1, 2, 3}

s.discard(2)
print(s)  # {1, 3}

# Discarding a non-existent element — no error
s.discard(99)
print(s)  # {1, 3} — still fine

### 5c. `pop()` — remove and return an arbitrary element

Removes and returns **some** element (you can't predict which one). Raises `KeyError` on an empty set.

Useful when you need to drain a set one element at a time.

In [ ]:
s = {"x", "y", "z"}

popped = s.pop()
print(f"Popped: {popped}")  # Could be any of x, y, z
print(f"Remaining: {s}")

# Pop from empty set → KeyError
empty = set()
try:
    empty.pop()
except KeyError as e:
    print(f"Can't pop from empty set: {e}")

### 5d. `clear()` — remove all elements

In [ ]:
s = {1, 2, 3, 4, 5}
s.clear()
print(s)       # set()
print(len(s))  # 0

### 5e. Removal methods comparison

| Method | Missing element behavior | Returns | Use when |
|--------|------------------------|---------|----------|
| `remove(x)` | Raises `KeyError` | `None` | You're certain `x` is in the set |
| `discard(x)` | Silently ignored | `None` | You're not sure if `x` exists |
| `pop()` | Raises `KeyError` if empty | The removed element | You need any arbitrary element |
| `clear()` | N/A | `None` | You want to empty the entire set |

---
## 6. Membership Testing — The Killer Feature

This is the **#1 reason** to use a set. The `in` operator checks membership in **O(1) average time** — compared to O(n) for lists. This makes sets the go-to for lookup-heavy problems.

In interviews, converting a list to a set for O(1) lookups is one of the most common optimization patterns.

In [ ]:
# Basic membership test
allowed = {"admin", "editor", "viewer"}

print("admin" in allowed)     # True
print("hacker" in allowed)    # False
print("hacker" not in allowed) # True

In [ ]:
# Performance comparison: list vs set lookup
import time

big_list = list(range(1_000_000))
big_set = set(range(1_000_000))
target = 999_999  # worst case for list

# List lookup — O(n), scans all elements
start = time.perf_counter()
_ = target in big_list
list_time = time.perf_counter() - start
print(f"List lookup: {list_time:.6f} seconds")

# Set lookup — O(1), direct hash
start = time.perf_counter()
_ = target in big_set
set_time = time.perf_counter() - start
print(f"Set lookup:  {set_time:.6f} seconds")

print(f"Set is ~{list_time / max(set_time, 1e-9):.0f}x faster")

### When to convert a list to a set for lookups

**Rule of thumb:** If you need to check membership **more than once**, convert to a set first.

```python
# BAD — O(n) per check, O(n*m) total
for item in items:
    if item in big_list:  # O(n) each time!
        ...

# GOOD — O(1) per check, O(n+m) total
big_set = set(big_list)   # O(n) one-time cost
for item in items:
    if item in big_set:   # O(1) each time!
        ...
```

---
## 7. Set Size and Emptiness

In [ ]:
s = {1, 2, 3}

# Length
print(len(s))  # 3

# Check if empty (Pythonic way)
empty = set()
if not empty:
    print("Set is empty")  # This prints

if s:
    print("Set is not empty")  # This prints

# Boolean value of sets
print(bool(set()))  # False — empty set is falsy
print(bool({1}))    # True — non-empty set is truthy

---
## 8. Iterating Over Sets

You can loop over a set with `for`, but remember: **order is not guaranteed**. Each run may produce a different order.

In [ ]:
colors = {"red", "green", "blue", "yellow"}

# Basic iteration
for color in colors:
    print(color)

# If you need sorted order, sort first
print("\nSorted:")
for color in sorted(colors):
    print(color)

In [ ]:
# Enumerate works with sets (but order is arbitrary)
s = {"a", "b", "c"}
for i, val in enumerate(s):
    print(f"{i}: {val}")

---
## 9. Set Operations — Union

**Union** combines all elements from both sets. Elements that appear in *either* set (or both) go into the result.

Think of it as **"everything from A plus everything from B"** with duplicates removed.

Two syntaxes: the `|` operator or the `.union()` method. The method can accept any iterable; the operator requires both sides to be sets.

In [ ]:
a = {1, 2, 3}
b = {3, 4, 5}

# Operator syntax
print(a | b)       # {1, 2, 3, 4, 5}

# Method syntax
print(a.union(b))  # {1, 2, 3, 4, 5}

# union() accepts any iterable
print(a.union([10, 20]))  # {1, 2, 3, 10, 20}

# Multiple unions
c = {5, 6, 7}
print(a | b | c)              # {1, 2, 3, 4, 5, 6, 7}
print(a.union(b, c))          # {1, 2, 3, 4, 5, 6, 7}

# Original sets are NOT modified
print(a)  # {1, 2, 3}

---
## 10. Set Operations — Intersection

**Intersection** returns only elements that appear in **both** sets.

Think of it as **"what do A and B have in common?"**

Two syntaxes: `&` operator or `.intersection()` method.

In [ ]:
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}

# Operator
print(a & b)               # {3, 4}

# Method
print(a.intersection(b))   # {3, 4}

# intersection() accepts any iterable
print(a.intersection([3, 4, 5, 6]))  # {3, 4}

# Multiple intersections
c = {4, 5, 6, 7}
print(a & b & c)               # {4}
print(a.intersection(b, c))    # {4}

# No common elements → empty set
print({1, 2} & {3, 4})  # set()

---
## 11. Set Operations — Difference

**Difference** returns elements that are in the first set but **not** in the second.

Think of it as **"what does A have that B doesn't?"**

**Order matters!** `A - B` is not the same as `B - A`.

In [ ]:
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}

# A - B: in A but not in B
print(a - b)               # {1, 2}
print(a.difference(b))     # {1, 2}

# B - A: in B but not in A (DIFFERENT!)
print(b - a)               # {5, 6}
print(b.difference(a))     # {5, 6}

# difference() accepts any iterable
print(a.difference([3, 4, 5, 6]))  # {1, 2}

# Multiple differences (A - B - C)
c = {1}
print(a - b - c)                # {2}
print(a.difference(b, c))       # {2}

---
## 12. Set Operations — Symmetric Difference

**Symmetric difference** returns elements that are in **either** set but **not both**. It's like union minus intersection.

Think of it as **"what's unique to each set?"** or **"XOR for sets"**.

In [ ]:
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}

# Operator
print(a ^ b)                          # {1, 2, 5, 6}

# Method
print(a.symmetric_difference(b))      # {1, 2, 5, 6}

# Symmetric difference is commutative (order doesn't matter)
print(a ^ b == b ^ a)  # True

# Verification: union minus intersection
print((a | b) - (a & b))  # {1, 2, 5, 6} — same result!

### Set operations visual summary

Given `A = {1, 2, 3, 4}` and `B = {3, 4, 5, 6}`:

| Operation | Symbol | Method | Result | Plain English |
|-----------|--------|--------|--------|---------------|
| Union | `A \| B` | `A.union(B)` | `{1,2,3,4,5,6}` | Everything from both |
| Intersection | `A & B` | `A.intersection(B)` | `{3,4}` | Only shared elements |
| Difference | `A - B` | `A.difference(B)` | `{1,2}` | In A but not B |
| Symmetric Diff | `A ^ B` | `A.symmetric_difference(B)` | `{1,2,5,6}` | In one but not both |

---
## 13. Subset, Superset, and Disjoint

These methods check **relationships** between two sets.

### 13a. Subset (`<=`, `issubset()`)

A is a subset of B if **every element of A is also in B**. Use `<` for a *proper* subset (A is a subset and A ≠ B).

In [ ]:
a = {1, 2}
b = {1, 2, 3, 4}

# Subset check
print(a <= b)          # True — a is a subset of b
print(a.issubset(b))   # True

# Proper subset (must be smaller)
print(a < b)   # True — a is a proper subset
print(a < a)   # False — a set is NOT a proper subset of itself
print(a <= a)  # True — a set IS a subset of itself

# Not a subset
c = {1, 5}
print(c <= b)  # False — 5 is not in b

### 13b. Superset (`>=`, `issuperset()`)

A is a superset of B if **A contains every element of B**. The reverse of subset.

In [ ]:
a = {1, 2, 3, 4}
b = {1, 2}

print(a >= b)            # True — a is a superset of b
print(a.issuperset(b))   # True

# Proper superset
print(a > b)   # True
print(a > a)   # False

### 13c. Disjoint (`isdisjoint()`)

Two sets are disjoint if they have **no elements in common**. Their intersection is empty.

In [ ]:
a = {1, 2, 3}
b = {4, 5, 6}
c = {3, 4, 5}

print(a.isdisjoint(b))  # True — no overlap
print(a.isdisjoint(c))  # False — 3 is in both

# Empty set is disjoint with everything
print(set().isdisjoint({1, 2, 3}))  # True

---
## 14. In-Place (Mutating) Set Operations

All four set operations have **in-place** versions that modify the original set instead of creating a new one.

| Operation | Operator | Method | Effect |
|-----------|----------|--------|--------|
| Union update | `A \|= B` | `A.update(B)` | A gets all elements from B |
| Intersection update | `A &= B` | `A.intersection_update(B)` | A keeps only elements also in B |
| Difference update | `A -= B` | `A.difference_update(B)` | A removes elements that are in B |
| Symmetric diff update | `A ^= B` | `A.symmetric_difference_update(B)` | A gets elements in one but not both |

In [ ]:
# |= (union update)
a = {1, 2, 3}
a |= {3, 4, 5}
print(f"|= result: {a}")  # {1, 2, 3, 4, 5}

# &= (intersection update)
a = {1, 2, 3, 4}
a &= {2, 3, 5}
print(f"&= result: {a}")  # {2, 3}

# -= (difference update)
a = {1, 2, 3, 4, 5}
a -= {3, 4}
print(f"-= result: {a}")  # {1, 2, 5}

# ^= (symmetric difference update)
a = {1, 2, 3}
a ^= {2, 3, 4}
print(f"^= result: {a}")  # {1, 4}

In [ ]:
# Method versions — same behavior
a = {1, 2, 3}
a.update([4, 5])                     # same as |=
print(a)  # {1, 2, 3, 4, 5}

a.intersection_update({2, 3, 4})     # same as &=
print(a)  # {2, 3, 4}

a.difference_update({3})             # same as -=
print(a)  # {2, 4}

a.symmetric_difference_update({4, 5}) # same as ^=
print(a)  # {2, 5}

---
## 15. Set Comprehensions

Set comprehensions work exactly like list comprehensions, but with curly braces `{}`. They automatically deduplicate results.

Syntax: `{expression for item in iterable if condition}`

In [ ]:
# Basic set comprehension
squares = {x ** 2 for x in range(10)}
print(squares)  # {0, 1, 4, 9, 16, 25, 36, 49, 64, 81}

# With filtering
even_squares = {x ** 2 for x in range(10) if x % 2 == 0}
print(even_squares)  # {0, 4, 16, 36, 64}

# Deduplication built-in
words = ["hello", "world", "hello", "python", "world"]
unique_lengths = {len(w) for w in words}
print(unique_lengths)  # {5, 6}

# Extract unique first characters
names = ["Alice", "Bob", "Anna", "Brian", "Charlie"]
first_chars = {name[0] for name in names}
print(first_chars)  # {'A', 'B', 'C'}

In [ ]:
# Practical: find unique vowels in a string
text = "The quick brown fox jumps over the lazy dog"
vowels = {c.lower() for c in text if c.lower() in "aeiou"}
print(vowels)  # {'a', 'e', 'i', 'o', 'u'}

# Practical: unique file extensions
files = ["app.py", "main.py", "style.css", "index.html", "test.py"]
extensions = {f.split(".")[-1] for f in files}
print(extensions)  # {'py', 'css', 'html'}

---
## 16. Frozenset — The Immutable Set

A `frozenset` is an **immutable** version of a set. You can't add, remove, or modify elements after creation.

Why does this exist?
1. **Frozensets are hashable** — you can use them as dict keys or put them inside other sets
2. **Thread safety** — immutable objects can be safely shared between threads
3. **Intentional immutability** — signals that this collection should not change

Frozensets support all **non-mutating** operations (union, intersection, etc.) but NOT add/remove/discard/pop/clear.

In [ ]:
# Create a frozenset
fs = frozenset([1, 2, 3, 2, 1])
print(fs)         # frozenset({1, 2, 3})
print(type(fs))   # <class 'frozenset'>

# All read operations work
print(2 in fs)        # True
print(len(fs))        # 3
print(fs | {4, 5})    # frozenset({1, 2, 3, 4, 5})
print(fs & {2, 3, 4}) # frozenset({2, 3})

# Mutation is NOT allowed
try:
    fs.add(4)
except AttributeError as e:
    print(f"Can't add: {e}")

In [ ]:
# Frozenset as a dict key (because it's hashable)
permissions = {
    frozenset({"read"}): "viewer",
    frozenset({"read", "write"}): "editor",
    frozenset({"read", "write", "delete"}): "admin",
}

user_perms = frozenset({"read", "write"})
print(permissions[user_perms])  # "editor"

# Frozenset inside a set (set of sets!)
groups = {frozenset({1, 2}), frozenset({3, 4}), frozenset({1, 2})}
print(groups)       # {frozenset({1, 2}), frozenset({3, 4})}
print(len(groups))  # 2 — duplicate removed

---
## 17. Copying Sets

Like other mutable collections, assigning a set to a new variable creates a **reference**, not a copy. Use `.copy()` or `set()` to make an independent shallow copy.

In [ ]:
# Reference (alias) — NOT a copy
original = {1, 2, 3}
alias = original
alias.add(99)
print(original)  # {1, 2, 3, 99} — original changed too!

# Shallow copy — independent
original = {1, 2, 3}
copy1 = original.copy()
copy2 = set(original)

copy1.add(99)
print(original)  # {1, 2, 3} — original unchanged
print(copy1)     # {1, 2, 3, 99}

---
## 18. When to Use Sets vs Lists vs Dicts

This is a common interview question and a key design decision in production code.

| Need | Best choice | Why |
|------|------------|-----|
| Ordered collection with duplicates | **List** | Preserves insertion order, allows duplicates |
| Fast membership testing | **Set** | O(1) lookup vs O(n) for lists |
| Remove duplicates | **Set** | Automatic deduplication |
| Key-value mapping | **Dict** | Maps unique keys to values |
| Counting occurrences | **Dict / Counter** | Need to track count per element |
| Need to index by position | **List** | Sets have no indices |
| Mathematical set operations | **Set** | Built-in union, intersection, difference |
| Immutable unique collection | **Frozenset** | Hashable, can be a dict key or set element |

In [ ]:
# Example: When a set is clearly better than a list

# Task: Find common elements between two collections
list_a = [1, 2, 3, 4, 5]
list_b = [4, 5, 6, 7, 8]

# BAD: nested loop O(n*m)
common_bad = [x for x in list_a if x in list_b]
print(common_bad)  # [4, 5]

# GOOD: set intersection O(n+m)
common_good = set(list_a) & set(list_b)
print(common_good)  # {4, 5}

---
## 19. Common Set Patterns

These patterns appear constantly in interviews and production code.

### 19a. Deduplication (preserving order)

Converting to a set removes duplicates but loses order. To deduplicate while **preserving order**, use a set as a seen-tracker.

In [ ]:
# Simple deduplication (order lost)
data = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3]
unique = list(set(data))
print(unique)  # Order not guaranteed

# Order-preserving deduplication (INTERVIEW PATTERN)
def deduplicate(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

print(deduplicate(data))  # [3, 1, 4, 5, 9, 2, 6] — order preserved!

### 19b. Finding duplicates

In [ ]:
# Find all elements that appear more than once
def find_duplicates(items):
    seen = set()
    dupes = set()
    for item in items:
        if item in seen:
            dupes.add(item)
        seen.add(item)
    return dupes

data = [1, 3, 5, 3, 7, 1, 9]
print(find_duplicates(data))  # {1, 3}

### 19c. Contains Duplicate (classic interview problem)

In [ ]:
# Check if any element appears more than once
def contains_duplicate(nums):
    return len(nums) != len(set(nums))

print(contains_duplicate([1, 2, 3, 4]))     # False
print(contains_duplicate([1, 2, 3, 1]))     # True

# Alternative: early exit with a set
def contains_duplicate_v2(nums):
    seen = set()
    for n in nums:
        if n in seen:
            return True
        seen.add(n)
    return False

print(contains_duplicate_v2([1, 2, 3, 4]))  # False
print(contains_duplicate_v2([1, 2, 3, 1]))  # True

### 19d. Two Sum lookup pattern

In [ ]:
# Check if any two numbers sum to a target
# (simplified version using a set — full version uses dict for indices)
def has_two_sum(nums, target):
    seen = set()
    for n in nums:
        complement = target - n
        if complement in seen:
            return True
        seen.add(n)
    return False

print(has_two_sum([2, 7, 11, 15], 9))   # True (2+7)
print(has_two_sum([2, 7, 11, 15], 10))  # False

### 19e. Intersection of arrays (interview classic)

In [ ]:
# Find common elements between two lists
def intersection(nums1, nums2):
    return list(set(nums1) & set(nums2))

print(intersection([1, 2, 2, 1], [2, 2]))       # [2]
print(intersection([4, 9, 5], [9, 4, 9, 8, 4]))  # [9, 4] or [4, 9]

### 19f. Finding missing elements

In [ ]:
# Find elements in required but not in current
required_fields = {"name", "email", "age", "phone"}
submitted_fields = {"name", "email"}

missing = required_fields - submitted_fields
print(f"Missing fields: {missing}")  # {'age', 'phone'}

extra = submitted_fields - required_fields
print(f"Extra fields: {extra}")  # set()

### 19g. Unique characters / Anagram detection

In [ ]:
# Check if a string has all unique characters
def all_unique(s):
    return len(s) == len(set(s))

print(all_unique("abcdef"))   # True
print(all_unique("hello"))    # False (two 'l's)

# Check if two strings are anagrams (same character set with same counts)
# Note: sets alone can't detect anagrams perfectly (they ignore frequency)
# But sets CAN quickly reject non-anagrams
def could_be_anagram(s1, s2):
    """Quick rejection: if character sets differ, not anagrams."""
    return set(s1) == set(s2) and len(s1) == len(s2)

print(could_be_anagram("listen", "silent"))  # True (still need freq check)
print(could_be_anagram("hello", "world"))    # False

### 19h. Longest Consecutive Sequence (interview hard)

In [ ]:
# Find the length of the longest consecutive element sequence
# This is an O(n) solution using a set — a famous interview problem

def longest_consecutive(nums):
    num_set = set(nums)
    longest = 0
    
    for n in num_set:
        # Only start counting from the beginning of a sequence
        if n - 1 not in num_set:  # This is a sequence start
            length = 1
            current = n
            while current + 1 in num_set:
                current += 1
                length += 1
            longest = max(longest, length)
    
    return longest

print(longest_consecutive([100, 4, 200, 1, 3, 2]))  # 4 (sequence: 1,2,3,4)
print(longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]))  # 9

---
## 20. Sets in Production Code

Real-world uses of sets in backend and data engineering.

In [ ]:
# --- Access Control ---
admin_perms = {"read", "write", "delete", "admin"}
editor_perms = {"read", "write"}
viewer_perms = {"read"}

def has_permission(user_perms, required_perms):
    """Check if user has all required permissions."""
    return required_perms <= user_perms  # subset check

print(has_permission(admin_perms, {"read", "write"}))  # True
print(has_permission(viewer_perms, {"read", "write"})) # False

# Extra permissions a user has beyond what's required
print(admin_perms - {"read", "write"})  # {'delete', 'admin'}

In [ ]:
# --- Feature Flags ---
enabled_features = {"dark_mode", "beta_search", "new_dashboard"}

def is_feature_enabled(feature):
    return feature in enabled_features  # O(1)

print(is_feature_enabled("dark_mode"))     # True
print(is_feature_enabled("old_feature"))   # False

In [ ]:
# --- API Input Deduplication ---
# User sends duplicate IDs in a batch request
requested_ids = [101, 102, 103, 101, 102, 104]
unique_ids = list(set(requested_ids))
print(f"Processing {len(unique_ids)} unique items: {unique_ids}")

# --- Schema Comparison ---
# Find columns that changed between two table versions
old_schema = {"id", "name", "email", "age"}
new_schema = {"id", "name", "email", "phone", "address"}

added = new_schema - old_schema
removed = old_schema - new_schema
unchanged = old_schema & new_schema

print(f"Added columns: {added}")      # {'phone', 'address'}
print(f"Removed columns: {removed}")  # {'age'}
print(f"Unchanged: {unchanged}")      # {'id', 'name', 'email'}

In [ ]:
# --- Rate Limiting with Unique IPs ---
class SimpleRateLimiter:
    def __init__(self):
        self.seen_ips = set()
    
    def check(self, ip):
        if ip in self.seen_ips:
            return False  # Already seen
        self.seen_ips.add(ip)
        return True
    
    def reset(self):
        self.seen_ips.clear()

limiter = SimpleRateLimiter()
print(limiter.check("192.168.1.1"))  # True (new IP)
print(limiter.check("192.168.1.1"))  # False (already seen)
print(limiter.check("10.0.0.1"))     # True (new IP)

---
## 21. Performance — Time Complexity

Sets are hash-table-based, giving them excellent average-case performance. The "Average" column is what you encounter in practice; "Worst" only happens with pathological hash collisions.

| Operation | Average | Worst | Notes |
|-----------|---------|-------|-------|
| `x in s` | O(1) | O(n) | Membership test — the killer feature |
| `s.add(x)` | O(1) | O(n) | Adding an element |
| `s.remove(x)` / `s.discard(x)` | O(1) | O(n) | Removing an element |
| `s.pop()` | O(1) | O(1) | Remove arbitrary element |
| `len(s)` | O(1) | O(1) | Length is cached |
| `s \| t` (union) | O(len(s) + len(t)) | | Creates new set |
| `s & t` (intersection) | O(min(len(s), len(t))) | | Iterates smaller set |
| `s - t` (difference) | O(len(s)) | | Iterates s, checks t |
| `s ^ t` (sym. diff) | O(len(s) + len(t)) | | Like union + membership |
| `s <= t` (subset) | O(len(s)) | | Check each element of s in t |
| Iteration | O(n) | O(n) | Visit each element |
| `set(iterable)` | O(n) | | Build from iterable |

**Key insight for interviews:** The O(1) membership test is what makes sets powerful. Converting a list to a set costs O(n) once, but then gives you unlimited O(1) lookups.

---
## 22. Built-in Functions That Work with Sets

In [ ]:
s = {5, 2, 8, 1, 9, 3}

print(len(s))       # 6
print(min(s))       # 1
print(max(s))       # 9
print(sum(s))       # 28
print(sorted(s))    # [1, 2, 3, 5, 8, 9] — returns a LIST

# any() / all()
print(any({0, 0, 1}))   # True (at least one truthy)
print(all({1, 2, 3}))   # True (all truthy)
print(all({0, 1, 2}))   # False (0 is falsy)

# Convert set to list/tuple
print(list(s))   # [order varies]
print(tuple(s))  # (order varies)

---
## 23. Common Gotchas and Edge Cases

These are the traps that trip up beginners and intermediates alike. Master these and you'll avoid the most common set bugs.

In [ ]:
# GOTCHA 1: {} is a dict, not a set
print(type({}))      # <class 'dict'>
print(type(set()))   # <class 'set'>

# GOTCHA 2: True/False and 1/0 are the same in sets
s = {True, 1, False, 0}
print(s)       # {True, False} or {0, 1}
print(len(s))  # 2 (not 4!)

# GOTCHA 3: Sets are unordered — no indexing
s = {1, 2, 3}
try:
    print(s[0])
except TypeError:
    print("Can't index a set!")

In [ ]:
# GOTCHA 4: Modifying a set during iteration
s = {1, 2, 3, 4, 5}
try:
    for x in s:
        if x % 2 == 0:
            s.remove(x)  # RuntimeError!
except RuntimeError as e:
    print(f"Error: {e}")

# FIX: Iterate over a copy or use comprehension
s = {1, 2, 3, 4, 5}
s = {x for x in s if x % 2 != 0}  # Keep only odds
print(s)  # {1, 3, 5}

# Alternative fix: iterate over a copy
s = {1, 2, 3, 4, 5}
for x in s.copy():
    if x % 2 == 0:
        s.remove(x)
print(s)  # {1, 3, 5}

In [ ]:
# GOTCHA 5: add() vs update() with strings
s1 = set()
s1.add("abc")      # Adds "abc" as ONE element
print(s1)           # {'abc'}

s2 = set()
s2.update("abc")   # Adds 'a', 'b', 'c' as THREE elements
print(s2)           # {'a', 'b', 'c'}

# GOTCHA 6: set() with a single string
s = set("hello")   # Each CHARACTER becomes an element
print(s)            # {'h', 'e', 'l', 'o'} — not {'hello'}!

In [ ]:
# GOTCHA 7: Set operations with operators require sets on both sides
s = {1, 2, 3}

# Method works with any iterable
print(s.union([4, 5]))        # {1, 2, 3, 4, 5} ✓
print(s.intersection([2, 3])) # {2, 3} ✓

# Operator requires set on BOTH sides
try:
    print(s | [4, 5])  # TypeError!
except TypeError as e:
    print(f"Operator error: {e}")

# Fix: convert to set first
print(s | set([4, 5]))  # {1, 2, 3, 4, 5} ✓

In [ ]:
# GOTCHA 8: Empty set comparison
s = set()
print(s == set())  # True
print(s == {})     # False! {} is a dict, not a set
print(set() == {}) # False!

# GOTCHA 9: Sets don't preserve insertion order
# (Unlike dicts which preserve insertion order since Python 3.7)
s = set()
for x in [5, 3, 1, 4, 2]:
    s.add(x)
print(s)  # Order NOT guaranteed to be [5, 3, 1, 4, 2]

---
## 24. Complete Set Methods Reference

Every method available on a Python `set` object:

| Method | Returns | Description |
|--------|---------|-------------|
| `s.add(x)` | `None` | Add element x |
| `s.remove(x)` | `None` | Remove x; KeyError if missing |
| `s.discard(x)` | `None` | Remove x; no error if missing |
| `s.pop()` | element | Remove and return arbitrary element |
| `s.clear()` | `None` | Remove all elements |
| `s.copy()` | `set` | Shallow copy |
| `s.update(t)` | `None` | Add all elements from iterable t |
| `s.union(t)` | `set` | New set with elements from s and t |
| `s.intersection(t)` | `set` | New set with elements common to s and t |
| `s.difference(t)` | `set` | New set with elements in s but not t |
| `s.symmetric_difference(t)` | `set` | New set with elements in s or t but not both |
| `s.intersection_update(t)` | `None` | Keep only elements found in both |
| `s.difference_update(t)` | `None` | Remove elements found in t |
| `s.symmetric_difference_update(t)` | `None` | Keep elements in either but not both |
| `s.issubset(t)` | `bool` | True if every element of s is in t |
| `s.issuperset(t)` | `bool` | True if every element of t is in s |
| `s.isdisjoint(t)` | `bool` | True if s and t have no common elements |

---
## Summary

Sets are your **go-to for uniqueness and fast lookups**. The key takeaways:

1. **O(1) membership testing** — convert lists to sets when you need fast `in` checks
2. **Automatic deduplication** — `set(list_with_dupes)` instantly removes duplicates
3. **Mathematical operations** — union, intersection, difference, symmetric difference
4. **Elements must be hashable** — no lists or dicts inside sets
5. **Unordered** — no indexing, no slicing, no guaranteed iteration order
6. **Frozenset** — immutable set that can be a dict key or set element
7. **Interview gold** — contains duplicate, two sum lookup, intersection of arrays, longest consecutive sequence

When you think "I need to check if something exists in a collection", think **set**.